In [ ]:
!pip install -q -U "crewai[google-genai]"

In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter Gemini API Key: ")

Enter Gemini API Key: ··········


In [ ]:
from crewai import Agent, Task, Crew, LLM
from crewai.tools import BaseTool
from pydantic import BaseModel, Field

llm = LLM(
    model="gemini/gemini-3.8-flash",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.3
)

print("Gemini initialized successfully!")

Gemini initialized successfully!


In [ ]:
class MovieInput(BaseModel):
    genre: str = Field(description="Movie genre")


class MovieTool(BaseTool):
    name: str = "Movie Recommendation Tool"

    description: str = "Returns movie recommendations based on genre."

    args_schema: type[BaseModel] = MovieInput

    def _run(self, genre: str) -> str:
        movies = {
            "action": "Avengers, John Wick, Mission Impossible",
            "comedy": "3 Idiots, Home Alone, The Mask",
            "romance": "Titanic, The Notebook, La La Land",
            "science fiction": "Interstellar, Inception, Avatar",
            "horror": "The Conjuring, IT, A Quiet Place"
        }

        return movies.get(
            genre.lower(),
            "Try exploring popular movies in your selected genre."
        )


movie_tool = MovieTool()

print("Movie tool created successfully!")

Movie tool created successfully!


In [ ]:
movie_agent = Agent(
    role="Movie Recommendation Expert",

    goal="Recommend interesting movies based on user preferences",

    backstory="You are a movie expert who helps users discover great movies.",

    llm=llm,

    tools=[movie_tool],

    verbose=True
)

print("Movie Agent created successfully!")

Movie Agent created successfully!


In [ ]:
movie_task = Task(
    description="""
    Recommend 5 science fiction movies.

    Use the Movie Recommendation Tool.

    Explain briefly why each movie is worth watching.
    """,

    expected_output="A list of 5 science fiction movies with short descriptions.",

    agent=movie_agent
)

print("Movie task created successfully!")

Movie task created successfully!


In [ ]:
crew = Crew(
    agents=[movie_agent],
    tasks=[movie_task],
    verbose=True
)

print("Crew created successfully!")

Crew created successfully!


In [ ]:
result = await crew.kickoff_async()

print("\n========== MOVIE RECOMMENDATIONS ==========")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 947eab2e-eb3f-4408-afc0-94505b1234d0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Recommend 5 science fiction movies.                                                                        │
│                                                                                                                 │
│      Use the Movie Recommendation Tool.                                                                         │
│                                                                                                                 │
│      Explain briefly why each movie is worth watching.                                                          │
│                                                                                                                 │
│  ID: b6975ebc-0b7f-4f93-869b-0d69b21eec41                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Movie Recommendation Expert                                                                             │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Recommend 5 science fiction movies.                                                                        │
│                                                                                                                 │
│      Use the Movie Recommendation Tool.                                                                         │
│                                                                                                                 │
│      Explain briefly why each movie is worth watching.                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: movie_recommendation_tool                                                                                │
│  Args: {'genre': 'science fiction'}                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: movie_recommendation_tool                                                                                │
│  Output: Interstellar, Inception, Avatar                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: movie_recommendation_tool                                                                                │
│  Args: {'genre': 'sci-fi'}                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (2 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Movie Recommendation Expert                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here are 5 science fiction movies that are well worth watching, including top picks from the Movie             │
│  Recommendation Tool:                                                                                           │
│                                                                                                                 │
│  1. **Interstellar (2014)**                                                                                     │
│     * **Why it's worth watching:** Directed by Christopher Nolan, this visually breathtaking space odyssey      │
│  explores human survival, black holes, and the physics of time dilation. Grounded in real theoretical           │
│  astrophysics and anchored by a powerful emotional core about love transcending dimensions, it also features a  │
│  phenomenal, haunting score by Hans Zimmer.                                                                     │
│                                                                                                                 │
│  2. **Inception (2010)**                                                                                        │
│     * **Why it's worth watching:** Another Christopher Nolan masterpiece, this film takes the classic heist     │
│  genre and transports it into the shared architecture of the human subconscious. With stunning practical        │
│  effects, an intricate multi-layered plot, and exhilarating action set pieces, it remains one of the most       │
│  inventive and thought-provoking blockbusters of the 21st century.                                              │
│                                                                                                                 │
│  3. **Avatar (2009)**                                                                                           │
│     * **Why it's worth watching:** James Cameron's groundbreaking epic revolutionized cinematic visual effects  │
│  and 3D technology. Beyond its technical achievements, it offers an astonishingly detailed and vibrant alien    │
│  world in Pandora, pairing an immersive visual spectacle with timeless themes of nature, connection, and        │
│  anti-colonialism.                                                                                              │
│                                                                                                                 │
│  4. **Blade Runner 2049 (2017)**                                                                                │
│     * **Why it's worth watching:** Directed by Denis Villeneuve, this neo-noir sequel expands brilliantly upon  │
│  Ridley Scott's 1982 classic. With Oscar-winning cinematography by Roger Deakins and a deliberate, atmospheric  │
│  pace, it offers a deeply philosophical meditation on artificial intelligence, memory, identity, and what       │
│  truly makes someone human.                                                                                     │
│                                                                                                                 │
│  5. **Arrival (2016)**                                                                                          │
│     * **Why it's worth watching:** Rather than relying on laser battles and hostile invasions, this cerebral    │
│  sci-fi drama centers on linguistics and communication 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Recommend 5 science fiction movies.                                                                        │
│                                                                                                                 │
│      Use the Movie Recommendation Tool.                                                                         │
│                                                                                                                 │
│      Explain briefly why each movie is worth watching.                                                          │
│                                                                                                                 │
│  Agent: Movie Recommendation Expert                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


========== MOVIE RECOMMENDATIONS ==========
Here are 5 science fiction movies that are well worth watching, including top picks from the Movie Recommendation Tool:

1. **Interstellar (2014)**
   * **Why it's worth watching:** Directed by Christopher Nolan, this visually breathtaking space odyssey explores human survival, black holes, and the physics of time dilation. Grounded in real theoretical astrophysics and anchored by a powerful emotional core about love transcending dimensions, it also features a phenomenal, haunting score by Hans Zimmer.

2. **Inception (2010)**
   * **Why it's worth watching:** Another Christopher Nolan masterpiece, this film takes the classic heist genre and transports it into the shared architecture of the human subconscious. With stunning practical effects, an intricate multi-layered plot, and exhilarating action set pieces, it remains one of the most inventive and thought-provoking blockbusters of the 21st century.

3. **Avatar (2009)**
   * **Why it's wo

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 947eab2e-eb3f-4408-afc0-94505b1234d0                                                                       │
│  Final Output: Here are 5 science fiction movies that are well worth watching, including top picks from the     │
│  Movie Recommendation Tool:                                                                                     │
│                                                                                                                 │
│  1. **Interstellar (2014)**                                                                                     │
│     * **Why it's worth watching:** Directed by Christopher Nolan, this visually breathtaking space odyssey      │
│  explores human survival, black holes, and the physics of time dilation. Grounded in real theoretical           │
│  astrophysics and anchored by a powerful emotional core about love transcending dimensions, it also features a  │
│  phenomenal, haunting score by Hans Zimmer.                                                                     │
│                                                                                                                 │
│  2. **Inception (2010)**                                                                                        │
│     * **Why it's worth watching:** Another Christopher Nolan masterpiece, this film takes the classic heist     │
│  genre and transports it into the shared architecture of the human subconscious. With stunning practical        │
│  effects, an intricate multi-layered plot, and exhilarating action set pieces, it remains one of the most       │
│  inventive and thought-provoking blockbusters of the 21st century.                                              │
│                                                                                                                 │
│  3. **Avatar (2009)**                                                                                           │
│     * **Why it's worth watching:** James Cameron's groundbreaking epic revolutionized cinematic visual effects  │
│  and 3D technology. Beyond its technical achievements, it offers an astonishingly detailed and vibrant alien    │
│  world in Pandora, pairing an immersive visual spectacle with timeless themes of nature, connection, and        │
│  anti-colonialism.                                                                                              │
│                                                                                                                 │
│  4. **Blade Runner 2049 (2017)**                                                                                │
│     * **Why it's worth watching:** Directed by Denis Villeneuve, this neo-noir sequel expands brilliantly upon  │
│  Ridley Scott's 1982 classic. With Oscar-winning cinematography by Roger Deakins and a deliberate, atmospheric  │
│  pace, it offers a deeply philosophical meditation on artificial intelligence, memory, identity, and what       │
│  truly makes someone human.                                                                                     │
│                                                                                                                 │
│  5. **Arrival (2016)**                                                                                          │
│     * **Why it's worth watching:** Rather than relying on laser battles and hostile invasions, this cerebral    │
│  sci-fi drama centers on linguistics and communication